# 01 · Capa Silver: compuestos anuales sin nubes con Spark

La Amazonía tiene nubosidad casi permanente: una sola escena casi nunca está despejada.
La solución estándar es el **compuesto por mediana**: para cada píxel se toma la mediana de
todas las observaciones válidas (sin nube ni sombra) de la temporada seca.

**Patrón Big Data aplicado:** *divide y vencerás por teselas*.

1. `binaryFile` carga los GeoTIFF como bytes en un DataFrame distribuido.
2. `groupBy(year, tile).applyInPandas(...)` envía a cada worker todas las escenas de una tesela.
3. Cada worker calcula el compuesto e índices con numpy y devuelve una fila por píxel.
4. El resultado se guarda en Delta particionado por año.

No hay ningún paso que traiga la imagen completa al driver: el mismo código escala de 2 500 km²
a todo Madre de Dios (85 000 km²) solo agregando workers.

In [0]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify
dbutils.library.restartPython()

In [0]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [0]:
from pyspark.sql import functions as F, types as T
from pathlib import Path
# Se importan funciones, no el módulo: en Databricks el objeto módulo arrastra el cargador interno
# del workspace (dbruntime), que no existe en los workers. Las funciones viajan serializadas por valor.
from geocloud.raster import (register_for_spark, composite_tile_to_pixels,
                             hansen_tile_to_pixels, geobosques_tile_to_pixels)
register_for_spark()

s2 = (spark.read.format("binaryFile").option("pathGlobFilter", "*.tif")
      .load(f"{VOL}/bronze/s2")
      .select("year", "tile", "content"))
# En serverless no se usa la API RDD: el conteo se hace con la API DataFrame
dbx.show(s2.groupBy("year").agg(F.countDistinct("tile").alias("teselas"), F.count("*").alias("archivos")).orderBy("year"))

## 1. Índices espectrales

| Índice | Fórmula (Sentinel-2) | Qué detecta |
|---|---|---|
| NDVI | (B8 − B4) / (B8 + B4) | vigor de la vegetación |
| NBR | (B8 − B12) / (B8 + B12) | remoción de cobertura |
| NDWI | (B3 − B8) / (B3 + B8) | agua abierta |
| MNDWI | (B3 − B11) / (B3 + B11) | agua turbia (pozas mineras) |
| BSI | ((B11 + B4) − (B8 + B2)) / ((B11 + B4) + (B8 + B2)) | suelo desnudo |

In [0]:
bands = [b.lower() for b in C.S2_BANDS]
idx = ["ndvi", "nbr", "ndwi", "mndwi", "bsi"]
schema = T.StructType(
    [T.StructField("tile", T.StringType()), T.StructField("year", T.IntegerType()),
     T.StructField("row", T.IntegerType()), T.StructField("col", T.IntegerType()),
     T.StructField("x", T.DoubleType()), T.StructField("y", T.DoubleType()),
     T.StructField("lon", T.DoubleType()), T.StructField("lat", T.DoubleType())]
    + [T.StructField(c, T.FloatType()) for c in bands + idx]
    + [T.StructField("n_obs", T.ShortType())])
cols = schema.fieldNames()

def composite_udf(pdf):
    out = composite_tile_to_pixels([bytes(c) for c in pdf["content"]])
    out.insert(0, "year", int(pdf["year"].iloc[0]))
    out.insert(0, "tile", str(pdf["tile"].iloc[0]))
    return out[cols]

pixels = s2.groupBy("year", "tile").applyInPandas(composite_udf, schema=schema)

In [0]:
%%time
dbx.save_table(pixels, "silver_s2_pixeles", partition_by="year")
silver = spark.table(dbx.tbl("silver_s2_pixeles"))
print(f"Píxeles Silver: {silver.count():,}")

In [0]:
dbx.show(silver.groupBy("year").agg(
    F.count("*").alias("pixeles"),
    F.round(F.avg("n_obs"), 2).alias("obs_validas_media"),
    F.round(F.avg("ndvi"), 3).alias("ndvi_medio"),
    F.round(F.avg(F.when((F.col("ndvi") < 0.3) & ((F.col("bsi") > 0) | (F.col("mndwi") > 0)), 1).otherwise(0)), 4)
      .alias("frac_perturbada")).orderBy("year"))

**Control de calidad:** el NDVI es un cociente normalizado y debe estar en [-1, 1]. Si un año tiene valores
fuera de rango, hay un problema de escala u offset en Bronze y el pipeline se detiene antes de llegar a Gold.

In [0]:
qc = (silver.groupBy("year")
      .agg(F.expr("percentile_approx(ndvi, array(0.01, 0.5, 0.99))").alias("ndvi_p01_p50_p99"),
           F.avg(F.when(F.abs("ndvi") > 1, 1).otherwise(0)).alias("frac_fuera_de_rango"))
      .orderBy("year"))
dbx.show(qc)
malos = [r.year for r in qc.collect() if r.frac_fuera_de_rango > 0.001]
assert not malos, f"NDVI fuera de [-1, 1] en {malos}: revisar escala/offset en Bronze (notebook 00)"
print("Control de calidad radiométrico: OK")

## 2. Hansen Global Forest Change en la misma grilla

`lossyear` codifica el año de pérdida de bosque (1 = 2001, ..., 24 = 2024; 0 = sin pérdida).
Se usa como **referencia independiente** para validar y como etiqueta del modelo de ML.

In [0]:
hansen_schema = T.StructType([T.StructField("row", T.IntegerType()), T.StructField("col", T.IntegerType()),
                              T.StructField("lon", T.DoubleType()), T.StructField("lat", T.DoubleType()),
                              T.StructField("lossyear", T.ShortType()), T.StructField("treecover2000", T.ShortType())])

def hansen_udf(it):
    for pdf in it:
        for c in pdf["content"]:
            yield hansen_tile_to_pixels(bytes(c))

hansen = (spark.read.format("binaryFile").option("pathGlobFilter", "*.tif").load(f"{VOL}/bronze/hansen")
          .select("content").mapInPandas(hansen_udf, schema=hansen_schema))
dbx.save_table(hansen, "silver_hansen_pixeles")
hs = spark.table(dbx.tbl("silver_hansen_pixeles"))
dbx.show(hs.filter("lossyear BETWEEN 1 AND 24").groupBy("lossyear").count().orderBy("lossyear"))

## 3. Geobosques (MINAM): la referencia oficial del Perú

Banda 1 `perdida_gb`: 1-24 = año de pérdida 2001-2024, 25 = pérdida antrópica 2025, 26 = pérdida natural 2025.
Banda 2 `cobertura_gb`: 1 hidrografía, 2 pérdida 2001-2024, 3 pérdida antrópica 2025, 4 bosque 2025, 5 no bosque 2000.
Banda 3 `riesgo_gb`: riesgo de deforestación 2018 (1 muy bajo ... 5 muy alto).

In [0]:
gb_schema = T.StructType([T.StructField("row", T.IntegerType()), T.StructField("col", T.IntegerType()),
                          T.StructField("lon", T.DoubleType()), T.StructField("lat", T.DoubleType()),
                          T.StructField("perdida_gb", T.ShortType()), T.StructField("cobertura_gb", T.ShortType()),
                          T.StructField("riesgo_gb", T.ShortType())])

def gb_udf(it):
    for pdf in it:
        for c in pdf["content"]:
            yield geobosques_tile_to_pixels(bytes(c))

if Path(f"{VOL}/bronze/geobosques").exists():
    gb = (spark.read.format("binaryFile").option("pathGlobFilter", "*.tif").load(f"{VOL}/bronze/geobosques")
          .select("content").mapInPandas(gb_udf, schema=gb_schema))
    dbx.save_table(gb, "silver_geobosques_pixeles")
    gbs = spark.table(dbx.tbl("silver_geobosques_pixeles"))
    dbx.show(gbs.filter("perdida_gb BETWEEN 1 AND 26").groupBy("perdida_gb").count().orderBy("perdida_gb"))

## 4. Mapas de NDVI por año (agregación distribuida a 200 m para visualizar)

In [0]:
import numpy as np
import matplotlib.pyplot as plt

f = 10   # 10 x 10 píxeles de 20 m = 200 m
coarse = (silver.groupBy("year", (F.col("row") / f).cast("int").alias("r"), (F.col("col") / f).cast("int").alias("c"))
          .agg(F.avg("ndvi").alias("ndvi"))).toPandas().pipe(dbx.clean)
grid = C.Grid()
years = sorted(coarse.year.unique())
fig, axes = plt.subplots(1, len(years), figsize=(5 * len(years), 3.6))
for ax, yv in zip(np.atleast_1d(axes), years):
    m = np.full((grid.height // f + 1, grid.width // f + 1), np.nan)
    d = coarse[coarse.year == yv]
    m[d.r, d.c] = d.ndvi
    im = ax.imshow(m, cmap="RdYlGn", vmin=0, vmax=0.9)
    ax.set_title(f"NDVI mediano, temporada seca {yv}"); ax.axis("off")
plt.colorbar(im, ax=axes, fraction=0.02)
plt.savefig(FIG / "01_ndvi_anual.png", dpi=150, bbox_inches="tight"); plt.show()